# Cell 1 – Imports & environment check

In [3]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")

import json
import random
import copy
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import torch

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, hamming_loss, multilabel_confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction,
    TrainerCallback,
)

from peft import LoraConfig, get_peft_model, TaskType

from tqdm import tqdm

import bpr_pipeline as bpr

DATA_ROOT = Path("../data").resolve()
assert DATA_ROOT.exists(), f"Expected data root at {DATA_ROOT}, check your cwd"

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
ADAPTER_PATH = DATA_ROOT / "qwen_bpr_classifier"
CHECKPOINT_DIR = DATA_ROOT / "training_checkpoints_classifier"

# All 10 redesign heuristics, in the same fixed order bpr_pipeline applies them in.
# Using bpr.APPLICATION_ORDER directly (rather than retyping the list) guarantees
# the label order here always matches what the executor (Cell 13) expects.
HEURISTIC_NAMES = bpr.APPLICATION_ORDER
NUM_LABELS = len(HEURISTIC_NAMES)

print("Imports OK")
print("HEURISTIC_NAMES:", HEURISTIC_NAMES)
print("DATA_ROOT:", DATA_ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("BF16 supported:", torch.cuda.is_bf16_supported())


Imports OK
HEURISTIC_NAMES: ['task_elimination', 'task_composition', 'resequencing', 'knock_out', 'parallelism', 'case_based_work', 'numerical_involvement', 'task_automation', 'trusted_party', 'extra_resources']
DATA_ROOT: C:\Users\yousu\Downloads\SAP\project\data
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Total VRAM (GB): 6.4
BF16 supported: True


# Cell 2 – Load dataset

In [4]:
TRAIN_DIR = DATA_ROOT / "processed" / "train"
EVAL_DIR = DATA_ROOT / "processed" / "eval"

def load_records(folder: Path):
    files = sorted(folder.glob("*.json"))
    if not files:
        raise FileNotFoundError(f"No processed files found under {folder}. Did notebook 5 finish?")
    records = []
    for fp in tqdm(files, desc=f"Loading {folder.name}"):
        with open(fp, "r", encoding="utf-8") as f:
            records.append(json.load(f))
    return records

all_train_records = load_records(TRAIN_DIR)
all_eval_records = load_records(EVAL_DIR)

print(f"Total train files: {len(all_train_records)}")
print(f"Total eval files:  {len(all_eval_records)}")

# Proven-working scale from the earlier classifier run -- short fixed-length text
# prompts (Cell 3/4) make this large a sample perfectly tractable on this GPU.
MAX_TRAIN = 20000
MAX_EVAL = 4000

random.seed(42)
random.shuffle(all_train_records)
random.shuffle(all_eval_records)

train_records = all_train_records[:MAX_TRAIN]
eval_records = all_eval_records[:MAX_EVAL]

print(f"Sampled train: {len(train_records)}")
print(f"Sampled eval:  {len(eval_records)}")


Loading eval: 100%|██████████| 22084/22084 [03:14<00:00, 113.48it/s]


Total train files: 88336
Total eval files:  22084
Sampled train: 20000
Sampled eval:  4000


# Cell 3 – Build prompt & labels (all 10 heuristics)

In [5]:
def record_to_prompt(record):
    process = record["as-is"]
    tasks = [pt["task"]["task_name"] for pt in sorted(process["process_task"], key=lambda x: x["order"])]
    gateways = [f'{g["gateway_type"]} ({g["name"]})' for g in process.get("gateways", [])]

    # bpr.compute_measures() computes the exact structural measures (parallelism,
    # level_of_control, cost_outlier_ratio, ...) that the ground-truth qualify_*()
    # rules threshold against to decide which heuristics apply. Including them
    # directly gives the classifier the same signal the rules use, instead of
    # making it re-derive that signal from task names alone -- much more learnable.
    try:
        measures = bpr.compute_measures(process)
        measure_lines = "\n".join(
            f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}"
            for k, v in measures.items()
        )
    except Exception:
        # A handful of AS-IS records can have graph quirks that trip compute_measures;
        # fall back to task/gateway text alone rather than failing the whole batch.
        measure_lines = "(measures unavailable)"

    prompt = f"""### Process Name
{process["process_name"]}

### Tasks
{" -> ".join(tasks)}

### Gateways
{", ".join(gateways) if gateways else "None"}

### Process Measures
{measure_lines}
"""
    return prompt.strip()


def record_to_labels(record):
    applied = {item["heuristicName"]: item["isApplied"] for item in record["redesignTrace"]}
    return [1.0 if applied.get(name, False) else 0.0 for name in HEURISTIC_NAMES]


def build_dataset(records):
    texts = [record_to_prompt(r) for r in tqdm(records, desc="Building prompts")]
    labels = [record_to_labels(r) for r in records]
    return Dataset.from_dict({"text": texts, "labels": labels})


train_ds_raw = build_dataset(train_records)
eval_ds_raw = build_dataset(eval_records)

print(train_ds_raw[0]["text"])
print()
print("Labels:", dict(zip(HEURISTIC_NAMES, train_ds_raw[0]["labels"])))


Building prompts: 100%|██████████| 4000/4000 [00:00<00:00, 6699.14it/s]

### Process Name
W2-P2(SDLC) (Copy)

### Tasks
ANALYSIS -> DEISGN -> CODE -> TEST -> IMPLEMENT

### Gateways
EXCLUSIVE (NO ERRORS)

### Process Measures
parallelism: 0.000
level_of_control: 0.000
level_of_authorization: 0.000
batch: 0.000
periodic: 0.000
process_contacts: 0.000
department_involvement: 0.600
department_share: 0.200
role_usage: 0.714
user_involvement: 1.400
process_hand_offs: 1.000
knock_outs: 0.000
managerial_layers: 0.833
it_automation: 0.000
it_comm: 1.000
process_versions: 1
process_size: 5
cost_outlier_ratio: 2.154
duration_outlier_ratio: 1.640
resequencing_available: 0.000

Labels: {'task_elimination': 0.0, 'task_composition': 0.0, 'resequencing': 0.0, 'knock_out': 0.0, 'parallelism': 0.0, 'case_based_work': 0.0, 'numerical_involvement': 1.0, 'task_automation': 0.0, 'trusted_party': 1.0, 'extra_resources': 1.0}


# Cell 4 – Tokenization

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Fixed-length, truncated tokenization -- no OOM risk, no "would it fit" surprises,
# no examples silently skipped. The prompt is a short structured summary (Cell 3),
# not a full JSON dump, so 384 tokens is generous headroom, not a tight squeeze.
MAX_LENGTH = 384

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

train_ds = train_ds_raw.map(tokenize_function, batched=True)
eval_ds = eval_ds_raw.map(tokenize_function, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(train_ds)


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 20000
})


# Cell 5 – Load Qwen + LoRA (sequence classification head)

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print()
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 2,171,648 || all params: 496,213,376 || trainable%: 0.4376

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


# Cell 6 – Classification metrics

In [8]:
def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    probabilities = torch.sigmoid(torch.tensor(logits)).numpy()
    predictions = (probabilities >= 0.5).astype(int)
    labels = labels.astype(int)

    micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(labels, predictions, average="micro", zero_division=0)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
    label_p, label_r, label_f1, _ = precision_recall_fscore_support(labels, predictions, average=None, zero_division=0)
    subset_accuracy = accuracy_score(labels, predictions)

    metrics = {
        "subset_accuracy": subset_accuracy,
        "micro_precision": micro_p, "micro_recall": micro_r, "micro_f1": micro_f1,
        "macro_precision": macro_p, "macro_recall": macro_r, "macro_f1": macro_f1,
    }
    for i, name in enumerate(HEURISTIC_NAMES):
        metrics[f"precision_{name}"] = label_p[i]
        metrics[f"recall_{name}"] = label_r[i]
        metrics[f"f1_{name}"] = label_f1[i]
    return metrics


# Cell 7 – Progress bar callback

In [9]:
class TqdmProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.progress = tqdm(total=state.max_steps, desc="Training")

    def on_step_end(self, args, state, control, **kwargs):
        self.progress.update(1)

    def on_train_end(self, args, state, control, **kwargs):
        self.progress.close()


# Cell 8 – Training arguments

In [10]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    bf16=torch.cuda.is_available(),
    fp16=False,
    dataloader_num_workers=0,  # >0 spawns subprocess workers, which reliably crash inside a Windows Jupyter kernel
    report_to="none",
    seed=42,
)

print("Training arguments set")
print("Effective train batch size:",
      training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)


Training arguments set
Effective train batch size: 8


# Cell 9 – Trainer

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[TqdmProgressCallback()],
)


# Cell 10 – Train model

In [12]:
print("Training started...")
train_result = trainer.train()
print("Training finished")
print(train_result.metrics)

with open(DATA_ROOT / "classifier_training_metrics.json", "w") as f:
    json.dump(train_result.metrics, f, indent=2, default=str)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Training started...


Training:   0%|          | 1/12500 [00:02<7:05:43,  2.04s/it]

Epoch,Training Loss,Validation Loss,Subset Accuracy,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1,Precision Task Elimination,Recall Task Elimination,F1 Task Elimination,Precision Task Composition,Recall Task Composition,F1 Task Composition,Precision Resequencing,Recall Resequencing,F1 Resequencing,Precision Knock Out,Recall Knock Out,F1 Knock Out,Precision Parallelism,Recall Parallelism,F1 Parallelism,Precision Case Based Work,Recall Case Based Work,F1 Case Based Work,Precision Numerical Involvement,Recall Numerical Involvement,F1 Numerical Involvement,Precision Task Automation,Recall Task Automation,F1 Task Automation,Precision Trusted Party,Recall Trusted Party,F1 Trusted Party,Precision Extra Resources,Recall Extra Resources,F1 Extra Resources
1,0.280258,0.132924,0.582000,0.908978,0.955656,0.931733,0.782261,0.763552,0.764120,1.000000,0.971530,0.985560,0.902174,0.451087,0.601449,0.736264,0.728261,0.732240,0.630237,0.570957,0.599134,0.804910,0.963402,0.877053,0.000000,0.000000,0.000000,0.967308,0.999338,0.983062,0.992389,0.997450,0.994913,0.890362,0.978749,0.932466,0.898972,0.974746,0.935327
2,0.221689,0.108182,0.633750,0.931391,0.949702,0.940458,0.891140,0.877264,0.878835,0.993811,1.000000,0.996896,0.906542,0.527174,0.666667,0.774194,0.782609,0.778378,0.633083,0.694719,0.662470,0.884498,0.900000,0.892182,0.906667,0.971429,0.937931,0.973623,0.977815,0.975715,0.994205,1.000000,0.997094,0.914771,0.964481,0.938969,0.930010,0.954411,0.942052
3,0.205330,0.103881,0.644500,0.925400,0.961672,0.943187,0.881963,0.909533,0.892903,1.000000,0.993772,0.996876,0.793333,0.646739,0.712575,0.757895,0.782609,0.770053,0.616945,0.853135,0.716066,0.898594,0.922680,0.910478,0.958333,0.985714,0.971831,0.968780,0.996689,0.982536,0.994205,1.000000,0.997094,0.927783,0.943837,0.935741,0.903758,0.970154,0.935780
4,0.180802,0.110021,0.645500,0.931312,0.956090,0.943538,0.891755,0.886750,0.888098,0.997338,1.000000,0.998667,0.815068,0.646739,0.721212,0.788889,0.771739,0.780220,0.650641,0.669967,0.660163,0.896933,0.919588,0.908119,0.970588,0.942857,0.956522,0.967597,0.998675,0.982891,0.995278,0.998179,0.996726,0.911748,0.965999,0.938090,0.923468,0.953755,0.938367
5,0.109350,0.123767,0.653250,0.935895,0.951625,0.943694,0.895567,0.890119,0.891405,0.998224,1.000000,0.999111,0.828571,0.630435,0.716049,0.787234,0.804348,0.795699,0.649123,0.671617,0.660178,0.906619,0.910825,0.908717,0.971831,0.985714,0.978723,0.969971,0.994702,0.982181,0.995280,0.998543,0.996909,0.921385,0.953552,0.937192,0.927430,0.951459,0.939291


Training: 100%|██████████| 12500/12500 [2:32:04<00:00,  1.37it/s]   


Training finished
{'train_runtime': 9124.2678, 'train_samples_per_second': 10.96, 'train_steps_per_second': 1.37, 'total_flos': 8.2962137088e+16, 'train_loss': 0.21929003701210023, 'epoch': 5.0}


# Cell 11 – Evaluate classifier & save model

In [13]:
print("Running evaluation...")
evaluation_metrics = trainer.evaluate()
for k, v in evaluation_metrics.items():
    print(f"{k:30}: {v}")

trainer.save_model(str(ADAPTER_PATH))
tokenizer.save_pretrained(str(ADAPTER_PATH))
print("Model saved to", ADAPTER_PATH)

with open(DATA_ROOT / "classifier_eval_metrics.json", "w") as f:
    json.dump(evaluation_metrics, f, indent=2, default=str)


Running evaluation...


Training Loss,Validation Loss,Epoch,Subset Accuracy,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1,Precision Task Elimination,Recall Task Elimination,F1 Task Elimination,Precision Task Composition,Recall Task Composition,F1 Task Composition,Precision Resequencing,Recall Resequencing,F1 Resequencing,Precision Knock Out,Recall Knock Out,F1 Knock Out,Precision Parallelism,Recall Parallelism,F1 Parallelism,Precision Case Based Work,Recall Case Based Work,F1 Case Based Work,Precision Numerical Involvement,Recall Numerical Involvement,F1 Numerical Involvement,Precision Task Automation,Recall Task Automation,F1 Task Automation,Precision Trusted Party,Recall Trusted Party,F1 Trusted Party,Precision Extra Resources,Recall Extra Resources,F1 Extra Resources
0.109350,0.123767,5,0.653250,0.935895,0.951625,0.943694,0.895567,0.890119,0.891405,0.998224,1.000000,0.999111,0.828571,0.630435,0.716049,0.787234,0.804348,0.795699,0.649123,0.671617,0.660178,0.906619,0.910825,0.908717,0.971831,0.985714,0.978723,0.969971,0.994702,0.982181,0.995280,0.998543,0.996909,0.921385,0.953552,0.937192,0.927430,0.951459,0.939291


eval_loss                     : 0.12376696616411209
eval_subset_accuracy          : 0.65325
eval_micro_precision          : 0.9358950899664532
eval_micro_recall             : 0.9516249069709749
eval_micro_f1                 : 0.9436944555490636
eval_macro_precision          : 0.8955666614696514
eval_macro_recall             : 0.8901194997733972
eval_macro_f1                 : 0.8914050699502992
eval_precision_task_elimination: 0.9982238010657194
eval_recall_task_elimination  : 1.0
eval_f1_task_elimination      : 0.9991111111111111
eval_precision_task_composition: 0.8285714285714286
eval_recall_task_composition  : 0.6304347826086957
eval_f1_task_composition      : 0.7160493827160493
eval_precision_resequencing   : 0.7872340425531915
eval_recall_resequencing      : 0.8043478260869565
eval_f1_resequencing          : 0.7956989247311828
eval_precision_knock_out      : 0.6491228070175439
eval_recall_knock_out         : 0.6716171617161716
eval_f1_knock_out             : 0.6601784266017843
eva

# Cell 12 – FlowAnalysisService (process metrics: cycle time, cost, ...)

In [14]:
# --- FlowAnalysisService: GraphBuilder / PathEnumerator / calculate_process_metrics ---
MAX_COMPOSITE_DEPTH = 3

def _task_times(task: dict):
    proc = task.get("expected_process_time") or 0
    wait = task.get("expected_waiting_time") or 0
    rework = task.get("expected_rework_time") or 0
    return float(proc), float(wait), float(rework)

def _task_cost(task: dict) -> float:
    proc, _wait, rework = _task_times(task)
    d_i_hours = (proc + rework) / 60.0
    cost = 0.0
    for job_task in task.get("jobTasks", []) or []:
        job = job_task.get("job") or {}
        hourly_rate = float(job.get("hourlyRate") or 0)
        alloc_pct = float(job_task.get("time_allocation_percentage") or 0)
        cost += d_i_hours * hourly_rate * (alloc_pct / 100.0)
    return cost

@dataclass
class TaskInfo:
    task_id: int
    order: int
    proc_time: float
    wait_time: float
    rework_time: float
    cost: float
    child_process_id: Optional[int] = None

def _build_task_index(process_json: dict):
    tasks = {}
    for pt in process_json.get("process_task", []) or []:
        task = pt.get("task")
        child_process_id = pt.get("child_process_id")
        if task is None:
            synthetic_id = -(child_process_id or pt.get("process_task_id"))
            tasks[synthetic_id] = TaskInfo(synthetic_id, pt.get("order", 0), 0.0, 0.0, 0.0, 0.0, child_process_id)
            continue
        proc, wait, rework = _task_times(task)
        tasks[task["task_id"]] = TaskInfo(
            task["task_id"], pt.get("order", 0), proc, wait, rework, _task_cost(task), child_process_id
        )
    return tasks

@dataclass
class Node:
    kind: str
    ref: Optional[int] = None
    label: Optional[str] = None
    terminal_after: bool = False

def _normalize_branch_probs(branches):
    raw = [float(b.get("probability") or 0) for b in branches]
    total = sum(raw)
    if total <= 0:
        n = len(branches) or 1
        return [1.0 / n] * len(branches)
    if abs(total - 1.0) < 1e-9:
        return raw
    return [p / total for p in raw]

def _resolve_branch_target(branch):
    if branch.get("target_task_id") is not None:
        return Node(kind="task", ref=branch["target_task_id"], terminal_after=bool(branch.get("connect_to_end")))
    if branch.get("target_gateway_id") is not None:
        return Node(kind="gateway", ref=branch["target_gateway_id"])
    return Node(kind="end", label=branch.get("end_event_name"))

class GraphBuilder:
    def __init__(self, process_json: dict):
        self.tasks = _build_task_index(process_json)
        self.gateways = process_json.get("gateways", []) or []
        self.gateway_by_id = {g["gateway_pk_id"]: g for g in self.gateways}
        self.gateway_by_after_task = {g["after_task_id"]: g for g in self.gateways if g.get("after_task_id") is not None}
        self._ordered_task_ids = sorted(self.tasks.keys(), key=lambda tid: self.tasks[tid].order)

    def find_start_node(self):
        targeted = set()
        for g in self.gateways:
            for b in g.get("branches", []):
                if b.get("target_gateway_id") is not None:
                    targeted.add(b["target_gateway_id"])
        qualifying = [g for g in self.gateways if g.get("after_task_id") is None and g["gateway_pk_id"] not in targeted]
        if qualifying:
            return Node(kind="gateway", ref=qualifying[0]["gateway_pk_id"])
        if self._ordered_task_ids:
            return Node(kind="task", ref=self._ordered_task_ids[0])
        return Node(kind="end", label=None)

    def next_after_task(self, task_id):
        gw = self.gateway_by_after_task.get(task_id)
        if gw is not None:
            return Node(kind="gateway", ref=gw["gateway_pk_id"])
        info = self.tasks.get(task_id)
        if info is None:
            # Dangling reference: some redesigned to-be records (from bpr_pipeline's
            # apply_elimination/apply_composition) can remove a task without updating
            # every gateway that still points at it. Treat as an early path end rather
            # than crashing the whole metrics computation.
            return Node(kind="end", label=None)
        order = info.order
        later = [tid for tid in self._ordered_task_ids if self.tasks[tid].order > order]
        if later:
            return Node(kind="task", ref=later[0])
        return Node(kind="end", label=None)

    def converge_target(self, gateway):
        if gateway.get("converge_at_task_id") is not None:
            return Node(kind="task", ref=gateway["converge_at_task_id"])
        if gateway.get("converge_at_gateway_id") is not None:
            return Node(kind="gateway", ref=gateway["converge_at_gateway_id"])
        if gateway.get("converge_to_end"):
            return Node(kind="end", label=gateway.get("converge_gateway_name"))
        return Node(kind="end", label=None)

@dataclass
class PathResult:
    probability: float
    task_ids: list = field(default_factory=list)
    end_label: Optional[str] = None

class PathEnumerator:
    def __init__(self, graph, child_processes=None, max_paths=5000):
        self.graph = graph
        self.child_processes = child_processes or {}
        self.max_paths = max_paths
        self.warnings = []

    def enumerate(self):
        start = self.graph.find_start_node()
        results = []
        self._walk(start, 1.0, [], frozenset(), 0, results)
        if not results:
            results.append(PathResult(probability=1.0, task_ids=[]))
        return results

    def _walk(self, node, probability, task_ids, visited_gateways, depth, results):
        if len(results) >= self.max_paths:
            self.warnings.append("max_paths limit reached; enumeration truncated")
            return
        if node.kind == "end":
            results.append(PathResult(probability=probability, task_ids=list(task_ids), end_label=node.label))
            return
        if node.kind == "task":
            task_ids = task_ids + [node.ref]
            nxt = Node(kind="end", label=None) if node.terminal_after else self.graph.next_after_task(node.ref)
            self._walk(nxt, probability, task_ids, visited_gateways, depth, results)
            return
        if node.kind == "gateway":
            gateway = self.graph.gateway_by_id.get(node.ref)
            if gateway is None:
                results.append(PathResult(probability=probability, task_ids=list(task_ids)))
                return
            if node.ref in visited_gateways:
                self.warnings.append(f"cyclic reference detected at gateway {node.ref}; loop truncated")
                results.append(PathResult(probability=probability, task_ids=list(task_ids)))
                return
            visited_gateways = visited_gateways | {node.ref}
            gtype = (gateway.get("gateway_type") or "EXCLUSIVE").upper()
            branches = gateway.get("branches", []) or []
            if not branches:
                results.append(PathResult(probability=probability, task_ids=list(task_ids)))
                return
            if gtype == "EXCLUSIVE":
                probs = _normalize_branch_probs(branches)
                for b, p in zip(branches, probs):
                    self._walk(_resolve_branch_target(b), probability * p, task_ids, visited_gateways, depth, results)
                return
            if gtype == "PARALLEL":
                merged = list(task_ids)
                for b in branches:
                    merged += self._collect_branch_tasks(_resolve_branch_target(b), gateway)
                self._walk(self.graph.converge_target(gateway), probability, merged, visited_gateways, depth, results)
                return
            if gtype == "INCLUSIVE":
                n = len(branches)
                probs = [float(b.get("probability") or 0) for b in branches]
                for mask in range(1, 1 << n):
                    subset = [i for i in range(n) if mask & (1 << i)]
                    subset_p = 1.0
                    for i in range(n):
                        subset_p *= probs[i] if i in subset else (1.0 - probs[i])
                    merged = list(task_ids)
                    for i in subset:
                        merged += self._collect_branch_tasks(_resolve_branch_target(branches[i]), gateway)
                    self._walk(self.graph.converge_target(gateway), probability * subset_p, merged, visited_gateways, depth, results)
                return
            probs = _normalize_branch_probs(branches)
            for b, p in zip(branches, probs):
                self._walk(_resolve_branch_target(b), probability * p, task_ids, visited_gateways, depth, results)

    def _collect_branch_tasks(self, node, owning_gateway, _depth=0):
        collected = []
        converge = self.graph.converge_target(owning_gateway)
        cur = node
        while _depth < 500:
            if cur.kind == "end":
                break
            if cur.kind == "task":
                if converge.kind == "task" and cur.ref == converge.ref:
                    break
                collected.append(cur.ref)
                cur = self.graph.next_after_task(cur.ref)
                _depth += 1
                continue
            if cur.kind == "gateway":
                if converge.kind == "gateway" and cur.ref == converge.ref:
                    break
                self.warnings.append(f"nested gateway {cur.ref} inside a parallel/inclusive branch was not expanded")
                break
        return collected

def _path_metrics(path, tasks):
    pt_k = wt_k = rt_k = c_k = 0.0
    for tid in path.task_ids:
        info = tasks.get(tid)
        if info is None:
            continue
        pt_k += info.proc_time
        wt_k += info.wait_time
        rt_k += info.rework_time
        c_k += info.cost
    return {"PT_k": pt_k, "WT_k": wt_k, "RT_k": rt_k, "D_k": pt_k + wt_k + rt_k, "C_k": c_k}

def calculate_process_metrics(process_json, child_processes=None, _depth=0):
    if _depth > MAX_COMPOSITE_DEPTH:
        return {"cycle_time_minutes": 0.0, "labor_cost_per_case": 0.0, "processing_time_minutes": 0.0,
                "waiting_time_minutes": 0.0, "rework_time_minutes": 0.0,
                "cycle_time_efficiency_percent": 0.0, "paths_evaluated": 0,
                "warnings": ["max composite sub-process depth (3) exceeded; returned zeros"]}

    graph = GraphBuilder(process_json)
    enumerator = PathEnumerator(graph, child_processes=child_processes)
    paths = enumerator.enumerate()
    warnings = list(enumerator.warnings)

    for tid, info in graph.tasks.items():
        if info.child_process_id is not None:
            child = (child_processes or {}).get(info.child_process_id)
            if child is not None:
                cr = calculate_process_metrics(child, child_processes=child_processes, _depth=_depth + 1)
                info.proc_time, info.wait_time = cr["processing_time_minutes"], cr["waiting_time_minutes"]
                info.rework_time, info.cost = cr["rework_time_minutes"], cr["labor_cost_per_case"]
            else:
                warnings.append(f"composite sub-process slot references child_process_id={info.child_process_id} "
                                 f"but no matching JSON was supplied; treated as zero-duration/zero-cost")

    e_ct = e_pt = e_wt = e_rt = e_cost = 0.0
    for path in paths:
        m = _path_metrics(path, graph.tasks)
        e_ct += path.probability * m["D_k"]; e_pt += path.probability * m["PT_k"]
        e_wt += path.probability * m["WT_k"]; e_rt += path.probability * m["RT_k"]
        e_cost += path.probability * m["C_k"]

    cte = (e_pt / e_ct * 100.0) if e_ct > 0 else 0.0
    return {"cycle_time_minutes": round(e_ct, 2), "labor_cost_per_case": round(e_cost, 2),
            "processing_time_minutes": round(e_pt, 2), "waiting_time_minutes": round(e_wt, 2),
            "rework_time_minutes": round(e_rt, 2), "cycle_time_efficiency_percent": round(cte, 2),
            "paths_evaluated": len(paths), "warnings": warnings}

print("FlowAnalysisService defined")


FlowAnalysisService defined


# Cell 13 – Executor: apply predicted heuristics deterministically

In [15]:
# The executor does NOT ask the model to write JSON. It takes the classifier's
# yes/no per heuristic and mechanically runs the SAME apply_* functions that built
# the training dataset in notebook 5 -- so the output is guaranteed schema-valid
# (barring the same rare pre-existing edge cases the original dataset build hit).
_by_name = {name: (hid, apply_fn) for hid, name, qualify, apply_fn in bpr.HEURISTICS}

def redesign_with_predicted_labels(as_is_record: dict, predicted_labels: dict) -> dict:
    working = copy.deepcopy(as_is_record)
    trace = []
    for name in bpr.APPLICATION_ORDER:
        hid, apply_fn = _by_name[name]
        if predicted_labels.get(name, False):
            working, targets, reason = apply_fn(working)
            applied = bool(targets)
            trace.append({
                "heuristicId": hid, "heuristicName": name, "isApplied": applied,
                "taskApplied": targets, "reasonApplied": reason if applied else None,
            })
        else:
            trace.append({
                "heuristicId": hid, "heuristicName": name, "isApplied": False,
                "taskApplied": [], "reasonApplied": None,
            })
    return {"as-is": as_is_record, "to-be": working, "redesignTrace": trace}

print("Executor defined -- reuses bpr_pipeline.HEURISTICS / APPLICATION_ORDER directly")


Executor defined -- reuses bpr_pipeline.HEURISTICS / APPLICATION_ORDER directly


# Cell 14 – Predict heuristics + run executor on eval set

In [16]:
model.eval()

@torch.no_grad()
def predict_labels_batch(texts, batch_size=16):
    all_probs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Predicting"):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, truncation=True, max_length=MAX_LENGTH,
                            padding=True, return_tensors="pt").to(model.device)
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits).float().cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs, axis=0)

eval_texts = [record_to_prompt(r) for r in eval_records]
eval_probs = predict_labels_batch(eval_texts)
eval_preds = (eval_probs >= 0.5).astype(int)

print(f"Predicted labels for {len(eval_records)} eval records")
print("Example predicted:    ", dict(zip(HEURISTIC_NAMES, eval_preds[0].tolist())))
print("Example ground truth: ", dict(zip(HEURISTIC_NAMES, record_to_labels(eval_records[0]))))

eval_results = []
executor_error_count = 0
for record, pred_row in tqdm(zip(eval_records, eval_preds), total=len(eval_records), desc="Running executor"):
    predicted_labels = {name: bool(pred_row[i]) for i, name in enumerate(HEURISTIC_NAMES)}

    result = {
        "process_code": record["as-is"]["process_code"],
        "predicted_labels": predicted_labels,
        "ground_truth_trace": record["redesignTrace"],
    }

    # The apply_* functions are well-guarded (checked in review), but this is the first
    # time they run against predicted label *combinations* the ground-truth generation
    # never produced, at full 20k/4k scale. One uncaught exception here would otherwise
    # kill the whole loop and lose all progress -- treat it as schema_ok=False instead.
    try:
        combined = redesign_with_predicted_labels(record["as-is"], predicted_labels)
        result["predicted_trace"] = combined["redesignTrace"]
        problems = bpr.validate_record(combined["to-be"])
        if problems:
            result["schema_ok"] = False
            result["validation_problems"] = problems
        else:
            result["schema_ok"] = True
            result["predicted_to_be"] = combined["to-be"]
    except Exception as exc:
        result["schema_ok"] = False
        result["executor_error"] = str(exc)
        executor_error_count += 1

    eval_results.append(result)

n = len(eval_results)
schema_ok_count = sum(r["schema_ok"] for r in eval_results)
print(f"Executor produced a schema-valid to-be for {schema_ok_count}/{n} eval records "
      f"({schema_ok_count/n:.1%})")
if executor_error_count:
    print(f"NOTE: {executor_error_count}/{n} records raised an exception during execution/validation "
          f"and were marked schema_ok=False rather than crashing the run.")


Predicting: 100%|██████████| 250/250 [02:00<00:00,  2.07it/s]


Predicted labels for 4000 eval records
Example predicted:     {'task_elimination': 0, 'task_composition': 0, 'resequencing': 0, 'knock_out': 0, 'parallelism': 1, 'case_based_work': 0, 'numerical_involvement': 0, 'task_automation': 0, 'trusted_party': 1, 'extra_resources': 1}
Example ground truth:  {'task_elimination': 0.0, 'task_composition': 0.0, 'resequencing': 0.0, 'knock_out': 0.0, 'parallelism': 1.0, 'case_based_work': 0.0, 'numerical_involvement': 0.0, 'task_automation': 0.0, 'trusted_party': 1.0, 'extra_resources': 1.0}


Running executor: 100%|██████████| 4000/4000 [00:08<00:00, 478.94it/s] 

Executor produced a schema-valid to-be for 4000/4000 eval records (100.0%)


# Cell 15 – Score cost/time reduction

In [17]:
def pct_reduction(before, after):
    return 0.0 if before == 0 else (before - after) / before * 100

for record, result in tqdm(zip(eval_records, eval_results), total=n, desc="Scoring metrics"):
    try:
        result["as_is_metrics"] = calculate_process_metrics(record["as-is"])
    except Exception as exc:
        result["as_is_metrics_error"] = str(exc)
        continue

    try:
        result["ground_truth_metrics"] = calculate_process_metrics(record["to-be"])
    except Exception as exc:
        result["ground_truth_metrics_error"] = str(exc)
        continue

    if result["schema_ok"]:
        try:
            result["predicted_metrics"] = calculate_process_metrics(result["predicted_to_be"])
        except Exception as exc:
            result["metrics_error"] = str(exc)
            result["schema_ok"] = False

gt_errors = sum(1 for r in eval_results if "ground_truth_metrics_error" in r)
if gt_errors:
    print(f"NOTE: {gt_errors}/{n} ground-truth to-be records errored during metrics computation and were skipped.")

scoreable = [r for r in eval_results if r.get("schema_ok") and "predicted_metrics" in r and "ground_truth_metrics" in r]
print(f"Scoreable on process metrics: {len(scoreable)}/{n}")

if scoreable:
    for r in scoreable:
        ct_before = r["as_is_metrics"]["cycle_time_minutes"]
        cost_before = r["as_is_metrics"]["labor_cost_per_case"]
        r["cycle_time_reduction_pred"] = pct_reduction(ct_before, r["predicted_metrics"]["cycle_time_minutes"])
        r["cycle_time_reduction_gt"] = pct_reduction(ct_before, r["ground_truth_metrics"]["cycle_time_minutes"])
        r["cost_reduction_pred"] = pct_reduction(cost_before, r["predicted_metrics"]["labor_cost_per_case"])
        r["cost_reduction_gt"] = pct_reduction(cost_before, r["ground_truth_metrics"]["labor_cost_per_case"])

    avg_ct_pred = float(np.mean([r["cycle_time_reduction_pred"] for r in scoreable]))
    avg_ct_gt = float(np.mean([r["cycle_time_reduction_gt"] for r in scoreable]))
    avg_cost_pred = float(np.mean([r["cost_reduction_pred"] for r in scoreable]))
    avg_cost_gt = float(np.mean([r["cost_reduction_gt"] for r in scoreable]))

    print(f"Avg cycle time reduction -- executed: {avg_ct_pred:.1f}%  ground truth: {avg_ct_gt:.1f}%  "
          f"(recovery: {avg_ct_pred/avg_ct_gt*100 if avg_ct_gt else 0:.0f}%)")
    print(f"Avg cost reduction       -- executed: {avg_cost_pred:.1f}%  ground truth: {avg_cost_gt:.1f}%  "
          f"(recovery: {avg_cost_pred/avg_cost_gt*100 if avg_cost_gt else 0:.0f}%)")
else:
    avg_ct_pred = avg_ct_gt = avg_cost_pred = avg_cost_gt = 0.0
    print("No scoreable records.")


Scoring metrics: 100%|██████████| 4000/4000 [00:01<00:00, 2221.70it/s]

Scoreable on process metrics: 4000/4000
Avg cycle time reduction -- executed: 19.9%  ground truth: 19.8%  (recovery: 101%)
Avg cost reduction       -- executed: 26.9%  ground truth: 26.8%  (recovery: 100%)


# Cell 16 – Score precision/recall/F1 on heuristics

In [18]:
y_true = np.array([record_to_labels(r) for r in eval_records])
y_pred = eval_preds

print(classification_report(y_true, y_pred, target_names=HEURISTIC_NAMES, zero_division=0, digits=3))

precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
hamming_acc = 1 - hamming_loss(y_true, y_pred)
exact_match = accuracy_score(y_true, y_pred)

print(f"Micro-avg -- Precision: {precision_micro:.3f}  Recall: {recall_micro:.3f}  F1: {f1_micro:.3f}")
print(f"Macro-avg -- Precision: {precision_macro:.3f}  Recall: {recall_macro:.3f}  F1: {f1_macro:.3f}")
print(f"Hamming accuracy (per-label):        {hamming_acc:.3f}")
print(f"Exact-match accuracy (whole trace):  {exact_match:.3f}")

matrices = multilabel_confusion_matrix(y_true, y_pred)
print("\nPer-heuristic confusion matrices:")
for name, cm in zip(HEURISTIC_NAMES, matrices):
    tn, fp, fn, tp = cm.ravel()
    print(f"{name:24s}  TP={tp:5d}  FP={fp:5d}  FN={fn:5d}  TN={tn:5d}")

aggregate_metrics = {
    "precision_micro": float(precision_micro), "recall_micro": float(recall_micro), "f1_micro": float(f1_micro),
    "precision_macro": float(precision_macro), "recall_macro": float(recall_macro), "f1_macro": float(f1_macro),
    "hamming_accuracy": float(hamming_acc), "exact_match_accuracy": float(exact_match),
}


                       precision    recall  f1-score   support

     task_elimination      0.998     1.000     0.999      1124
     task_composition      0.823     0.630     0.714       184
         resequencing      0.787     0.804     0.796        92
            knock_out      0.650     0.665     0.657       606
          parallelism      0.908     0.911     0.909      1940
      case_based_work      0.972     0.986     0.979        70
numerical_involvement      0.970     0.995     0.982      3020
      task_automation      0.995     0.999     0.997      2745
        trusted_party      0.921     0.953     0.937      3294
      extra_resources      0.928     0.952     0.940      3049

            micro avg      0.936     0.951     0.944     16124
            macro avg      0.895     0.889     0.891     16124
         weighted avg      0.936     0.951     0.943     16124
          samples avg      0.932     0.946     0.931     16124

Micro-avg -- Precision: 0.936  Recall: 0.951  F1: 0.

# Cell 17 – Save final summary

In [19]:
summary = {
    "model_name": MODEL_NAME,
    "heuristic_names": HEURISTIC_NAMES,
    "train_sample_size": len(train_records),
    "eval_sample_size": len(eval_records),
    "training_metrics": train_result.metrics,
    "classifier_eval_metrics": evaluation_metrics,
    "schema_ok_rate": schema_ok_count / n,
    "scoreable_count": len(scoreable),
    "avg_cycle_time_reduction_executed": avg_ct_pred,
    "avg_cycle_time_reduction_gt": avg_ct_gt,
    "avg_cost_reduction_executed": avg_cost_pred,
    "avg_cost_reduction_gt": avg_cost_gt,
    "label_metrics": aggregate_metrics,
}

with open(DATA_ROOT / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("=" * 50)
print("RUN COMPLETE -- see data/training_summary.json for full details")
print("=" * 50)
for k, v in summary.items():
    if k not in ("training_metrics", "classifier_eval_metrics"):
        print(f"{k}: {v}")


RUN COMPLETE -- see data/training_summary.json for full details
model_name: Qwen/Qwen2.5-0.5B
heuristic_names: ['task_elimination', 'task_composition', 'resequencing', 'knock_out', 'parallelism', 'case_based_work', 'numerical_involvement', 'task_automation', 'trusted_party', 'extra_resources']
train_sample_size: 20000
eval_sample_size: 4000
schema_ok_rate: 1.0
scoreable_count: 4000
avg_cycle_time_reduction_executed: 19.892916029590737
avg_cycle_time_reduction_gt: 19.784213494752255
avg_cost_reduction_executed: 26.877388612346664
avg_cost_reduction_gt: 26.755518249155536
label_metrics: {'precision_micro': 0.9361689143833527, 'recall_micro': 0.9514388489208633, 'f1_micro': 0.9437421180523515, 'precision_macro': 0.8952114522282686, 'recall_macro': 0.889494670816857, 'f1_macro': 0.8909990134416128, 'hamming_accuracy': 0.954275, 'exact_match_accuracy': 0.65175}
